In [1]:
# ============================================
# ЯЧЕЙКА 1: Импорты и настройка окружения (ОПТИМИЗИРОВАННАЯ ДЛЯ GPU)
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, recall_score,
    precision_recall_curve, confusion_matrix, classification_report,
    roc_curve, average_precision_score, log_loss, brier_score_loss
)

import lightgbm as lgb
import xgboost as xgb

# CatBoost опционально
try:
    import catboost as cb
    CATBOOST_AVAILABLE = True
    print("✅ CatBoost доступен")
except ImportError:
    CATBOOST_AVAILABLE = False
    print("⚠️ CatBoost не установлен (опционально)")

import joblib
import os
import json
from tqdm.notebook import tqdm
import time
import gc
from collections import defaultdict
from functools import partial

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR, OneCycleLR
import copy

# Настройки для производительности
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Проверка GPU
gpu_available = torch.cuda.is_available()
if gpu_available:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA версия: {torch.version.cuda}")
    print(f"   GPU память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не обнаружен, работа на CPU")

print(f"✅ LightGBM {lgb.__version__} | XGBoost {xgb.__version__}")
print(f"✅ GPU доступен для ML: {gpu_available}")

✅ CatBoost доступен
✅ GPU: NVIDIA GeForce RTX 3060
   CUDA версия: 12.6
   GPU память: 12.9 GB
✅ LightGBM 4.6.0 | XGBoost 3.2.0
✅ GPU доступен для ML: True


In [2]:
# ============================================
# ЯЧЕЙКА 2: КОНФИГУРАЦИЯ КОТЛА БКЗ-420-140 (РАСШИРЕННАЯ)
# ============================================

# Датчики температуры поверхностей нагрева
SENSORS = {
    '10HAH01CT103': {'surface_type': 'средние ширмы', 'material': '12Х1МФ', 'stage': 2, 'stream': 1},
    '10HAH01CT102': {'surface_type': 'средние ширмы', 'material': '12Х1МФ', 'stage': 2, 'stream': 1},
    '10HAH12CT101': {'surface_type': 'ширмы', 'material': '12Х1МФ', 'stage': 2, 'stream': 1},
    '10HAH12CT104': {'surface_type': 'ширмы', 'material': '12Х1МФ', 'stage': 2, 'stream': 4},
    '10HAH12CT110': {'surface_type': 'пароперегреватель', 'material': '12Х18Н12Т', 'stage': 3, 'stream': 2},
    '10HAH12CT108': {'surface_type': 'пароперегреватель', 'material': '12Х18Н12Т', 'stage': 3, 'stream': 2},
    '10HAH12CT106': {'surface_type': 'пароперегреватель', 'material': '12Х18Н12Т', 'stage': 3, 'stream': 2},
    '10HAH11CT114': {'surface_type': 'пароперегреватель', 'material': '12Х18Н12Т', 'stage': 4, 'stream': 1},
    '10HAH11CT113': {'surface_type': 'пароперегреватель', 'material': '12Х18Н12Т', 'stage': 4, 'stream': 1},
    '10HAH12CT116': {'surface_type': 'ширмы', 'material': '12Х1МФ', 'stage': 4, 'stream': 2},
    '10HAH12CT117': {'surface_type': 'ширмы', 'material': '12Х1МФ', 'stage': 4, 'stream': 2}
}

# Параметры поверхностей нагрева
SURFACE_PARAMS = {
    'средние ширмы': {
        'initial_thickness': 4.0, 'critical_thickness': 2.5,
        'normal_temp': 450, 'max_temp': 560,
        'A_constant': 3.5e-15, 'C_larson_miller': 20,
        'activation_energy': 450000, 'stress_mpa': 60,
        'repair_restoration': {'капитальный': 0.95, 'средний': 0.85, 'текущий': 0.70}
    },
    'ширмы': {
        'initial_thickness': 4.5, 'critical_thickness': 2.8,
        'normal_temp': 460, 'max_temp': 570,
        'A_constant': 3.5e-15, 'C_larson_miller': 20,
        'activation_energy': 450000, 'stress_mpa': 65,
        'repair_restoration': {'капитальный': 0.95, 'средний': 0.85, 'текущий': 0.70}
    },
    'пароперегреватель': {
        'initial_thickness': 5.0, 'critical_thickness': 3.0,
        'normal_temp': 480, 'max_temp': 580,
        'A_constant': 5.0e-14, 'C_larson_miller': 22,
        'activation_energy': 480000, 'stress_mpa': 55,
        'repair_restoration': {'капитальный': 0.95, 'средний': 0.85, 'текущий': 0.70}
    }
}

# История ремонтов (из ППР)

REPAIR_HISTORY = {
    'капитальный': [pd.Timestamp('2017-06-30 ')],
    'средний': [pd.Timestamp('2016-07-20 ')],
    'текущие': [
        pd.Timestamp('2014-09-07 '),
        pd.Timestamp('2014-10-30 '),
        pd.Timestamp('2015-09-19'),
        pd.Timestamp('2016-08-29 '),
        pd.Timestamp('2017-03-11 '),
        pd.Timestamp('2017-08-25 '),
        pd.Timestamp('2018-10-27 ')
    ]
}
FORECAST_HOURS = 24
STEPS_FORWARD = FORECAST_HOURS * 6
MODELS_DIR = "trained_models_corrected"
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"✅ Котёл БКЗ-420-140: {len(SENSORS)} датчиков | Прогноз: {FORECAST_HOURS}ч")

✅ Котёл БКЗ-420-140: 11 датчиков | Прогноз: 24ч


In [3]:
# ============================================
# ЯЧЕЙКА 3: ФИЗИЧЕСКАЯ МОДЕЛЬ ДЕГРАДАЦИИ (РАСШИРЕННАЯ)
# ============================================

class PhysicsBasedDegradation:
    """
    Физическая модель деградации металла поверхностей нагрева.
    Основана на:
    - Уравнении Ларсона-Миллера (LMP = T*(C + log(t)))
    - Уравнении Аррениуса для скорости ползучести
    - Модели накопления повреждений Качанова-Работнова
    """
    
    def __init__(self, surface_type):
        self.params = SURFACE_PARAMS[surface_type]
        self.H0 = self.params['initial_thickness']
        self.H_crit = self.params['critical_thickness']
        self.A = self.params['A_constant']
        self.C = self.params['C_larson_miller']
        self.Q = self.params['activation_energy']
        self.sigma = self.params['stress_mpa'] * 1e6
        self.R = 8.314
        self.T_norm = self.params['normal_temp'] + 273.15
    
    def larson_miller_parameter(self, temperature, hours):
        T_k = temperature + 273.15
        return T_k * (self.C + np.log10(hours + 1)) / 1000
    
    def calculate_wear_rate(self, temperature):
        """Скорость износа по Аррениусу, мм/час"""
        T_k = np.clip(np.atleast_1d(temperature) + 273.15, 273, 2000)
        rate_norm = self.A * np.exp(-self.Q / (self.R * self.T_norm)) * (self.sigma ** 2)
        rate = self.A * np.exp(-self.Q / (self.R * T_k)) * (self.sigma ** 2)
        rate_normalized = rate / rate_norm * 0.0001
        return float(np.clip(rate_normalized, 0, 0.01).item())
    
    def calculate_thickness(self, temperature_history, repairs=None):
        """Расчет текущей толщины стенки с учетом деградации и ремонтов"""
        dt = 1/6  # 10 минут в часах
        
        if isinstance(temperature_history, pd.Series):
            temp_values = temperature_history.values
            temp_index = temperature_history.index
        else:
            temp_values = np.array(temperature_history)
            temp_index = None
        
        n = len(temp_values)
        thickness = np.zeros(n)
        thickness[0] = self.H0
        current_thickness = self.H0
        
        for i in range(1, n):
            wear_rate = self.calculate_wear_rate(temp_values[i])
            current_thickness -= wear_rate * dt
            
            if repairs and temp_index is not None:
                current_date = temp_index[i]
                for repair_type, dates in repairs.items():
                    for repair_date in dates:
                        if current_date.date() == repair_date.date():
                            restoration = self.params['repair_restoration'].get(repair_type, 0.85)
                            current_thickness += (self.H0 - current_thickness) * restoration
            
            current_thickness = np.clip(current_thickness, 0, self.H0)
            thickness[i] = current_thickness
        
        return thickness
    
    def calculate_reliability(self, thickness):
        """Коэффициент надежности R(t) ∈ [0, 1]"""
        k = 8
        ratio = thickness / self.H_crit
        reliability = 1 / (1 + np.exp(-k * (ratio - 1.1)))
        return np.clip(reliability, 0, 1)
    
    def predict_time_to_failure(self, current_thickness, avg_temperature):
        """Прогноз времени до аварийного останова"""
        if current_thickness <= self.H_crit:
            return 0
        wear_rate = self.calculate_wear_rate(avg_temperature)
        if wear_rate <= 0:
            return float('inf')
        return (current_thickness - self.H_crit) / wear_rate


class CorrectedFeatureCreator:
    """
    Генератор признаков БЕЗ УТЕЧКИ ДАННЫХ.
    Все признаки, зависящие от времени, сдвинуты на FORECAST_HOURS назад.
    """
    
    def __init__(self, forecast_hours=24):
        self.forecast_hours = forecast_hours
        self.shift_steps = forecast_hours * 6
    
    def create_all_features(self, data, sensor_name):
        if sensor_name not in data.columns:
            return None
        
        features = pd.DataFrame(index=data.index)
        idx = data.index
        shift = self.shift_steps
        
        surface_type = SENSORS[sensor_name]['surface_type']
        degradation_model = PhysicsBasedDegradation(surface_type)
        
        temperature = data[sensor_name].astype(float)
        temp_values = temperature.values
        
        # 1. Темпоральные признаки (не сдвигаются - они известны заранее)
        features['hour_sin'] = np.sin(2 * np.pi * idx.hour / 24)
        features['hour_cos'] = np.cos(2 * np.pi * idx.hour / 24)
        features['month_sin'] = np.sin(2 * np.pi * idx.month / 12)
        features['month_cos'] = np.cos(2 * np.pi * idx.month / 12)
        features['dayofweek_sin'] = np.sin(2 * np.pi * idx.dayofweek / 7)
        features['dayofweek_cos'] = np.cos(2 * np.pi * idx.dayofweek / 7)
        features['hour'] = idx.hour
        features['month'] = idx.month
        features['quarter'] = idx.quarter
        features['dayofweek'] = idx.dayofweek
        features['is_weekend'] = (idx.dayofweek >= 5).astype(int)
        features['is_heating_season'] = ((idx.month >= 10) | (idx.month <= 4)).astype(int)
        features['is_night'] = ((idx.hour >= 22) | (idx.hour <= 5)).astype(int)
        features['is_morning_peak'] = ((idx.hour >= 6) & (idx.hour <= 9)).astype(int)
        features['is_evening_peak'] = ((idx.hour >= 17) & (idx.hour <= 20)).astype(int)
        
        # 2. Температурные признаки (СДВИНУТЫ)
        features['temp'] = np.roll(temp_values, shift)
        features['temp'][:shift] = temp_values[0]
        
        temp_series = pd.Series(temp_values)
        features['temp_ma_6h'] = temp_series.rolling(36, min_periods=1).mean().shift(shift).ffill().values
        features['temp_ma_24h'] = temp_series.rolling(144, min_periods=1).mean().shift(shift).ffill().values
        features['temp_ma_168h'] = temp_series.rolling(1008, min_periods=1).mean().shift(shift).ffill().values
        features['temp_std_24h'] = temp_series.rolling(144, min_periods=1).std().shift(shift).ffill().values
        
        # 3. Физические признаки деградации (СДВИНУТЫ)
        # Скорость износа
        wear_rate = np.array([degradation_model.calculate_wear_rate(t) for t in temp_values])
        features['wear_rate'] = np.roll(wear_rate, shift)
        features['wear_rate'][:shift] = wear_rate[0]
        
        # Толщина стенки
        thickness = degradation_model.calculate_thickness(temperature, REPAIR_HISTORY)
        thickness_shifted = np.roll(thickness, shift)
        thickness_shifted[:shift] = degradation_model.H0
        features['thickness'] = thickness_shifted
        features['thickness_initial'] = degradation_model.H0
        features['thickness_critical'] = degradation_model.H_crit
        features['thickness_ratio'] = thickness_shifted / degradation_model.H0
        
        # Надежность
        reliability = degradation_model.calculate_reliability(thickness)
        reliability_shifted = np.roll(reliability, shift)
        reliability_shifted[:shift] = 1.0
        features['reliability'] = reliability_shifted
        
        # 4. Признаки ремонтов (СДВИНУТЫ)
        all_repairs = sorted([r for r in 
            REPAIR_HISTORY['капитальный'] + 
            REPAIR_HISTORY['средний'] + 
            REPAIR_HISTORY['текущие'] 
            if r <= data.index.max()])
        
        days_since_repair = np.zeros(len(data.index))
        last_repair = None
        
        for i, dt in enumerate(data.index):
            if last_repair is not None:
                days_since_repair[i] = (dt - last_repair).total_seconds() / 86400
            for rp in all_repairs:
                if rp.date() == dt.date():
                    last_repair = rp
                    days_since_repair[i] = 0
                    break
        
        features['days_since_last_repair'] = np.roll(days_since_repair, shift)
        features['days_since_last_repair'][:shift] = 0
        
        # 5. Очистка
        features = features.replace([np.inf, -np.inf], np.nan)
        features = features.ffill().bfill().fillna(0)
        
        return features


feature_creator = CorrectedFeatureCreator(FORECAST_HOURS)
print("✅ Физическая модель деградации (Ларсон-Миллер + Аррениус)")
print(f"   Сдвиг признаков: {STEPS_FORWARD} шагов ({FORECAST_HOURS} часов) - защита от data leak")

✅ Физическая модель деградации (Ларсон-Миллер + Аррениус)
   Сдвиг признаков: 144 шагов (24 часов) - защита от data leak


In [4]:
# ============================================
# ЯЧЕЙКА 4: ВСЕ ML МОДЕЛИ (РАСШИРЕННЫЙ НАБОР)
# ============================================

def get_all_ml_models():
    """
    Расширенный набор моделей машинного обучения:
    - Ансамблевые методы (Bagging, Boosting, Stacking)
    - Базовые классификаторы
    - Градиентный бустинг на GPU
    """
    models = {}
    
    # 1. Bagging ансамбли
    models['Bagging_DecisionTree'] = {
        'model': BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=10, class_weight='balanced'),
            n_estimators=50, random_state=42, n_jobs=-1
        ),
        'params': {
            'n_estimators': [30, 50, 100],
            'max_samples': [0.7, 0.8, 1.0]
        }
    }
    
    # 2. RandomForest (оптимизированный)
    models['RandomForest'] = {
        'model': RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight='balanced',
            min_samples_leaf=20, min_samples_split=50, max_features='sqrt'
        ),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, 20],
            'min_samples_leaf': [10, 20, 50]
        }
    }
    
    # 3. ExtraTrees (оптимизированный)
    models['ExtraTrees'] = {
        'model': ExtraTreesClassifier(
            random_state=42, n_jobs=-1, class_weight='balanced',
            min_samples_leaf=20, min_samples_split=50, max_features='sqrt'
        ),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, 20],
            'min_samples_leaf': [10, 20, 50]
        }
    }
    
    # 4. GradientBoosting
    models['GradientBoosting'] = {
        'model': GradientBoostingClassifier(
            random_state=42, subsample=0.8, max_features='sqrt',
            min_samples_leaf=20, min_samples_split=50
        ),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [4, 6, 8],
            'learning_rate': [0.05, 0.1, 0.2]
        }
    }
    
    # 5. XGBoost с GPU
    models['XGBoost'] = {
        'model': xgb.XGBClassifier(
            random_state=42, n_jobs=-1, verbosity=0,
            tree_method='hist', device='cuda' if gpu_available else 'cpu',
            eval_metric='logloss'
        ),
        'params': {
            'max_depth': [4, 6, 8],
            'learning_rate': [0.05, 0.1, 0.2],
            'n_estimators': [100, 200, 300],
            'subsample': [0.7, 0.8, 1.0],
            'colsample_bytree': [0.7, 0.8, 1.0]
        }
    }
    
    # 6. LightGBM с GPU
    models['LightGBM'] = {
        'model': lgb.LGBMClassifier(
            random_state=42, n_jobs=-1, verbose=-1,
            device='gpu' if gpu_available else 'cpu',
            num_leaves=31, learning_rate=0.1
        ),
        'params': {
            'num_leaves': [31, 63, 127],
            'learning_rate': [0.05, 0.1, 0.2],
            'max_depth': [5, 7, 10],
            'min_child_samples': [20, 30, 50],
            'n_estimators': [100, 200, 300]
        },
        'fit_params': {'eval_metric': 'logloss'}
    }
    
    # 7. CatBoost с GPU
    try:
        models['CatBoost'] = {
            'model': cb.CatBoostClassifier(
                random_seed=42, verbose=False, thread_count=-1,
                task_type='GPU' if gpu_available else 'CPU',
                logging_level='Silent'
            ),
            'params': {
                'depth': [4, 6, 8],
                'learning_rate': [0.05, 0.1, 0.2],
                'iterations': [100, 200, 300],
                'l2_leaf_reg': [1, 3, 5]
            }
        }
    except:
        print("⚠️ CatBoost не установлен, пропущен")
    
       # 8. AdaBoost (ИСПРАВЛЕНАЯ ВЕРСИЯ)
    models['AdaBoost'] = {
        'model': AdaBoostClassifier(
            random_state=42
            # Параметр 'algorithm' удален, т.к. в современных версиях sklearn
            # используется только 'SAMME' (по умолчанию) и 'SAMME.R'
        ),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.5, 1.0, 1.5]
        }
    }
    
    # 9. Базовые классификаторы
    models['LogisticRegression'] = {
        'model': LogisticRegression(
            random_state=42, max_iter=1000, n_jobs=-1,
            class_weight='balanced', solver='liblinear'
        ),
        'params': {
            'C': [0.01, 0.1, 1.0, 10.0],
            'penalty': ['l2']
        }
    }
    
    models['DecisionTree'] = {
        'model': DecisionTreeClassifier(
            random_state=42, class_weight='balanced'
        ),
        'params': {
            'max_depth': [5, 10, 15, 20],
            'min_samples_leaf': [10, 20, 50],
            'min_samples_split': [20, 50, 100]
        }
    }
    
    models['KNN'] = {
        'model': KNeighborsClassifier(n_jobs=-1),
        'params': {
            'n_neighbors': [3, 5, 7, 11],
            'weights': ['uniform', 'distance'],
            'p': [1, 2]
        }
    }
    
    # 10. SVM (только для небольших данных, может быть медленным)
    models['SVM'] = {
        'model': SVC(
            random_state=42, class_weight='balanced',
            probability=True, max_iter=1000
        ),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'gamma': ['scale', 'auto'],
            'kernel': ['rbf']
        }
    }
    
    return models


device = torch.device('cuda' if gpu_available else 'cpu')
print(f"✅ ML модели и нейросети готовы | Устройство: {device}")

✅ ML модели и нейросети готовы | Устройство: cuda


In [5]:
# ============================================
# ЯЧЕЙКА 5: НЕЙРОСЕТЕВЫЕ АРХИТЕКТУРЫ (РАСШИРЕННЫЕ)
# ============================================

class SimpleDNN(nn.Module):
    """Простая полносвязная сеть с BatchNorm и Dropout"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)


class DeepDNN(nn.Module):
    """Глубокая полносвязная сеть (6 слоев)"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 16), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)


class WideDeepNet(nn.Module):
    """Wide & Deep архитектура"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.wide = nn.Linear(input_dim, 32)
        self.deep = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.combined = nn.Sequential(
            nn.Linear(96, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        wide_out = self.wide(x)
        deep_out = self.deep(x)
        combined = torch.cat([wide_out, deep_out], dim=1)
        return self.combined(combined)


class AttentionNet(nn.Module):
    """Сеть с Self-Attention механизмом"""
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, 
                                               dropout=dropout_rate, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x = self.input_proj(x).unsqueeze(1)
        x, attn_weights = self.attention(x, x, x)
        x = self.norm(x.squeeze(1))
        return self.fc(x), attn_weights


class ResidualNet(nn.Module):
    """Сеть с Residual блоками"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        
        self.res_block1 = nn.Sequential(
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 128), nn.BatchNorm1d(128)
        )
        
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        
        self.res_block2 = nn.Sequential(
            nn.Linear(64, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 64), nn.BatchNorm1d(64)
        )
        
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, x):
        x = self.dropout(torch.relu(self.bn1(self.fc1(x))))
        residual = x
        x = self.res_block1(x)
        x = torch.relu(x + residual)
        x = self.dropout(torch.relu(self.bn2(self.fc2(x))))
        residual = x
        x = self.res_block2(x)
        x = torch.relu(x + residual)
        x = torch.relu(self.bn3(self.fc3(x)))
        return self.sigmoid(self.fc4(x))


class MultiTaskNet(nn.Module):
    """Многозадачная сеть: классификация + регрессия"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.ReLU()
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
        # Regression head (для толщины стенки)
        self.regressor = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        shared_out = self.shared(x)
        cls_out = self.classifier(shared_out)
        reg_out = self.regressor(shared_out)
        return cls_out, reg_out


class LSTMPredictor(nn.Module):
    """LSTM для временных рядов"""
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout_rate=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, 
                            batch_first=True, dropout=dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        lstm_out, (hn, cn) = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.fc(self.dropout(last_out))


class ReliabilityNet(nn.Module):
    """Специализированная сеть для прогноза надежности"""
    def __init__(self, input_dim, dropout_rate=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)


# Словарь всех нейросетевых моделей
NEURAL_MODELS = {
    'SimpleDNN': SimpleDNN,
    'DeepDNN': DeepDNN,
    'WideDeepNet': WideDeepNet,
    'AttentionNet': AttentionNet,
    'ResidualNet': ResidualNet,
    'MultiTaskNet': MultiTaskNet,
    'LSTMPredictor': LSTMPredictor,
    'ReliabilityNet': ReliabilityNet
}

print(f"✅ Нейросетевые архитектуры готовы: {len(NEURAL_MODELS)} моделей")

✅ Нейросетевые архитектуры готовы: 8 моделей


In [6]:
# ============================================
# ЯЧЕЙКА 6: ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ============================================

print("📁 Загрузка данных...")
data_path = "temperature_data.csv"

use_cols = ['Date_Time'] + list(SENSORS.keys())
df = pd.read_csv(data_path, parse_dates=['Date_Time'], usecols=use_cols)
df.set_index('Date_Time', inplace=True)

# Очистка данных
for sensor in SENSORS:
    if sensor in df.columns:
        df.loc[df[sensor] < 0, sensor] = np.nan
        df.loc[df[sensor] > 600, sensor] = np.nan
        df[sensor] = df[sensor].interpolate(limit_direction='both').fillna(df[sensor].median())

df = df.resample('10min').mean()
print(f"📊 {len(df):,} записей | {df.index.min()} - {df.index.max()}")

# ХРОНОЛОГИЧЕСКОЕ РАЗДЕЛЕНИЕ (строго по времени!)
train_data = df[df.index.year <= 2015].copy()
val_data = df[(df.index.year >= 2016) & (df.index.year <= 2017)].copy()
test_data = df[df.index.year == 2018].copy()

print(f"📊 Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}")

📁 Загрузка данных...
📊 303,473 записей | 2013-01-01 00:00:00 - 2018-10-09 10:40:00
📊 Train: 157,680 | Val: 105,264 | Test: 40,529


In [7]:
# ============================================
# ЯЧЕЙКА 7: ЦЕЛЕВАЯ ПЕРЕМЕННАЯ (ЧЕСТНАЯ, БЕЗ УТЕЧКИ)
# ============================================

def create_realistic_target(data):
    target = pd.DataFrame(index=data.index)
    
    for sensor in SENSORS:
        if sensor in data.columns:
            surface_type = SENSORS[sensor]['surface_type']
            max_temp = SURFACE_PARAMS[surface_type]['max_temp']
            target[sensor] = (data[sensor] <= (max_temp - 20)).astype(int)
    
    target['is_working'] = target.min(axis=1)
    return target

train_target = create_realistic_target(train_data)
val_target = create_realistic_target(val_data)
test_target = create_realistic_target(test_data)

train_data['is_working'] = train_target['is_working'].shift(-STEPS_FORWARD)
val_data['is_working'] = val_target['is_working'].shift(-STEPS_FORWARD)
test_data['is_working'] = test_target['is_working'].shift(-STEPS_FORWARD)

print(f"✅ Целевая переменная создана (прогноз на {FORECAST_HOURS}ч вперед)")
print(f"⚖️ Работающих: train={train_data['is_working'].mean():.1%}, test={test_data['is_working'].mean():.1%}")


✅ Целевая переменная создана (прогноз на 24ч вперед)
⚖️ Работающих: train=50.3%, test=53.3%


In [8]:
# ============================================
# ЯЧЕЙКА 8: ОБУЧЕНИЕ ML МОДЕЛЕЙ (ОПТИМИЗИРОВАННАЯ - БЕЗ ЛИШНИХ ТЕСТОВ)
# ============================================

def get_ml_models_final():
    """
    Набор моделей для обучения
    """
    models = {}
    
    # 1. LightGBM GPU
    models['LightGBM_GPU'] = {
        'model': lgb.LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            device='gpu' if gpu_available else 'cpu',
            num_leaves=31
        ),
        'params': {
            'num_leaves': [31, 63],
            'learning_rate': [0.05, 0.1],
            'n_estimators': [100, 150]
        }
    }
    
    # 2. XGBoost GPU
    models['XGBoost_GPU'] = {
        'model': xgb.XGBClassifier(
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            tree_method='hist',
            device='cuda' if gpu_available else 'cpu',
            eval_metric='logloss'
        ),
        'params': {
            'max_depth': [4, 6],
            'learning_rate': [0.05, 0.1],
            'n_estimators': [100, 150]
        }
    }
    
    # 3. CatBoost GPU (если доступен)
    if CATBOOST_AVAILABLE:
        models['CatBoost_GPU'] = {
            'model': cb.CatBoostClassifier(
                random_seed=42,
                verbose=False,
                task_type='GPU' if gpu_available else 'CPU',
                iterations=100
            ),
            'params': {
                'depth': [4, 6],
                'learning_rate': [0.05, 0.1],
                'iterations': [100, 150]
            }
        }
    
    # 4. RandomForest CPU
    models['RandomForest'] = {
        'model': RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        ),
        'params': {
            'n_estimators': [100, 150],
            'max_depth': [10, 15]
        }
    }
    
    # 5. ExtraTrees CPU
    models['ExtraTrees'] = {
        'model': ExtraTreesClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        ),
        'params': {
            'n_estimators': [100, 150],
            'max_depth': [10, 15]
        }
    }
    
    return models


def find_optimal_threshold(y_val, y_val_pred):
    """Поиск оптимального порога по F1 на валидации"""
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, y_val_pred)
    f1_scores = 2 * precision_vals[:-1] * recall_vals[:-1] / (precision_vals[:-1] + recall_vals[:-1] + 1e-8)
    best_idx = np.argmax(f1_scores)
    threshold = thresholds[best_idx] if len(thresholds) > 0 else 0.5
    best_f1 = f1_scores[best_idx]
    return threshold, best_f1


print("="*80)
print("🚀 ОБУЧЕНИЕ ML МОДЕЛЕЙ (3 ФОЛДА - ТОЛЬКО ВАЛИДАЦИЯ)")
print("="*80)
print(f"📊 GPU: {'✅ ДОСТУПЕН' if gpu_available else '❌ НЕТ'}")
print(f"📊 Кросс-валидация: 3 фолда")
print(f"📊 Выбор лучшей модели: по ROC-AUC на валидации")
print(f"📊 Тестирование: ТОЛЬКО для лучшей модели")
print("="*80)

all_ml_results = {}
training_summary = []

for sensor_name in tqdm(SENSORS.keys(), desc="📡 Обучение датчиков"):
    if sensor_name not in df.columns:
        continue
    
    print(f"\n{'='*60}")
    print(f"📡 ДАТЧИК: {sensor_name}")
    print(f"   Тип: {SENSORS[sensor_name]['surface_type']}")
    print(f"{'='*60}")
    sensor_start = time.time()
    
    try:
        # Генерация признаков
        print("   ⚙️ Генерация признаков...")
        X_train = feature_creator.create_all_features(train_data, sensor_name)
        X_val = feature_creator.create_all_features(val_data, sensor_name)
        X_test = feature_creator.create_all_features(test_data, sensor_name)
        
        if X_train is None:
            print("   ❌ Ошибка: признаки не созданы")
            continue
        
        # Целевые переменные
        y_train = train_data['is_working']
        y_val = val_data['is_working']
        y_test = test_data['is_working']
        
        # Удаление NaN
        valid_train = y_train.notna()
        valid_val = y_val.notna()
        valid_test = y_test.notna()
        
        X_train = X_train[valid_train]
        y_train = y_train[valid_train]
        X_val = X_val[valid_val]
        y_val = y_val[valid_val]
        X_test = X_test[valid_test]
        y_test = y_test[valid_test]
        
        print(f"   📊 Train: {len(X_train):,} ({y_train.mean():.1%} раб.)")
        print(f"   📊 Val:   {len(X_val):,} ({y_val.mean():.1%} раб.)")
        print(f"   📊 Test:  {len(X_test):,} ({y_test.mean():.1%} раб.)")
        
        if len(X_train) < 500:
            print("   ⚠️ Пропуск: недостаточно данных")
            continue
        
        # Масштабирование
        print("   ⚙️ Масштабирование...")
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_val_s = scaler.transform(X_val)
        X_te_s = scaler.transform(X_test)
        
        # Обучение моделей и выбор лучшей по валидации
        print("\n   🚀 Обучение моделей (оценка только на валидации)...")
        print("   " + "-"*50)
        
        val_results = {}  # Храним результаты только на валидации
        
        models_dict = get_ml_models_final()
        
        for name, config in models_dict.items():
            try:
                t0 = time.time()
                
                # 3-фолд кросс-валидация на трейне для выбора гиперпараметров
                cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
                grid = GridSearchCV(
                    config['model'], 
                    config['params'], 
                    cv=cv, 
                    scoring='roc_auc', 
                    n_jobs=1,
                    verbose=0
                )
                
                print(f"   🔄 {name}...", end=' ', flush=True)
                grid.fit(X_tr_s, y_train)
                model = grid.best_estimator_
                
                # Оценка ТОЛЬКО на ВАЛИДАЦИИ (для выбора лучшей модели)
                y_val_pred = model.predict_proba(X_val_s)[:, 1]
                val_auc = roc_auc_score(y_val, y_val_pred)
                
                # Поиск оптимального порога на валидации
                threshold, val_f1 = find_optimal_threshold(y_val, y_val_pred)
                
                elapsed = time.time() - t0
                
                # Сохраняем результаты ТОЛЬКО валидации
                val_results[name] = {
                    'model': model,
                    'threshold': threshold,
                    'val_auc': val_auc,
                    'val_f1': val_f1,
                    'time': elapsed,
                    'best_params': grid.best_params_
                }
                
                # Маркер GPU/CPU
                marker = "🟢" if "GPU" in name else "🟡"
                
                print(f"{marker}")
                print(f"      ✅ Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f}")
                print(f"      ⏱️ Время: {elapsed:.1f} сек")
                print()
                
            except Exception as e:
                print(f"   ❌ {name}: {str(e)[:60]}\n")
                continue
        
        # ВЫБОР ЛУЧШЕЙ МОДЕЛИ ПО ROC-AUC НА ВАЛИДАЦИИ
        if val_results:
            print("   " + "-"*50)
            print("   📊 ВЫБОР ЛУЧШЕЙ МОДЕЛИ ПО VALIDATION AUC:")
            print("   " + "-"*50)
            
            # Сортируем по val_auc
            sorted_models = sorted(val_results.items(), key=lambda x: x[1]['val_auc'], reverse=True)
            
            for i, (name, results) in enumerate(sorted_models, 1):
                marker = "🏆" if i == 1 else "  "
                print(f"   {marker} {i}. {name:18s}: Val AUC={results['val_auc']:.4f} | Val F1={results['val_f1']:.4f}")
            
            # Лучшая модель
            best_name = sorted_models[0][0]
            best_results = sorted_models[0][1]
            
            print(f"\n   🏆 ЛУЧШАЯ МОДЕЛЬ: {best_name}")
            print(f"      Val AUC: {best_results['val_auc']:.4f}")
            print(f"      Val F1:  {best_results['val_f1']:.4f}")
            print(f"      Threshold: {best_results['threshold']:.3f}")
            
            # ТОЛЬКО ТЕПЕРЬ - ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ НА ТЕСТОВЫХ ДАННЫХ
            print(f"\n   🧪 ФИНАЛЬНОЕ ТЕСТИРОВАНИЕ НА ОТЛОЖЕННОЙ ВЫБОРКЕ (ТОЛЬКО ДЛЯ ЛУЧШЕЙ МОДЕЛИ):")
            
            # Переобучаем лучшую модель на ВСЕХ данных (train + val) для финального теста
            print(f"      ⚙️ Дообучение на train+val...")
            X_full = np.vstack([X_tr_s, X_val_s])
            y_full = np.concatenate([y_train.values, y_val.values])
            
            # Получаем конфиг лучшей модели
            best_config = models_dict[best_name]
            best_model_full = best_config['model'].__class__(**best_config['model'].get_params())
            best_model_full.set_params(**best_results['best_params'])
            
            # Обучаем на полных данных
            best_model_full.fit(X_full, y_full)
            
            # Финальное предсказание на тесте
            y_test_pred_final = best_model_full.predict_proba(X_te_s)[:, 1]
            y_test_cls_final = (y_test_pred_final >= best_results['threshold']).astype(int)
            
            final_auc = roc_auc_score(y_test, y_test_pred_final)
            final_f1 = f1_score(y_test, y_test_cls_final)
            final_acc = accuracy_score(y_test, y_test_cls_final)
            
            print(f"      ✅ ФИНАЛЬНЫЕ МЕТРИКИ НА ТЕСТЕ:")
            print(f"         AUC: {final_auc:.4f}")
            print(f"         F1:  {final_f1:.4f}")
            print(f"         Accuracy: {final_acc:.4f}")
            
            # Сохраняем модель
            model_data = {
                'model': best_model_full,
                'scaler': scaler,
                'threshold': float(best_results['threshold']),
                'model_name': best_name,
                'surface_type': SENSORS[sensor_name]['surface_type'],
                'metrics': {
                    'val_auc': float(best_results['val_auc']),
                    'val_f1': float(best_results['val_f1']),
                    'test_auc': float(final_auc),
                    'test_f1': float(final_f1),
                    'test_accuracy': float(final_acc)
                },
                'best_params': best_results['best_params']
            }
            
            path = os.path.join(MODELS_DIR, f"{sensor_name}.pkl")
            joblib.dump(model_data, path)
            
            total_time = time.time() - sensor_start
            
            all_ml_results[sensor_name] = model_data
            training_summary.append({
                'Датчик': sensor_name,
                'Тип': SENSORS[sensor_name]['surface_type'],
                'Модель': best_name,
                'Val_AUC': f"{best_results['val_auc']:.4f}",
                'Test_AUC': f"{final_auc:.4f}",
                'Test_F1': f"{final_f1:.4f}",
                'Время_сек': f"{total_time:.1f}"
            })
            
            print(f"\n   💾 Модель сохранена: {path}")
            print(f"   ⏱️ Общее время: {total_time:.1f} сек")
            
        else:
            print("   ❌ Не удалось обучить ни одной модели")
        
        # Очистка памяти
        del X_train, X_val, X_test, X_tr_s, X_val_s, X_te_s
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"   ❌ Критическая ошибка: {str(e)[:150]}")
        continue

# ИТОГОВЫЕ РЕЗУЛЬТАТЫ
print("\n" + "="*80)
print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ ОБУЧЕНИЯ")
print("="*80)

if training_summary:
    summary_df = pd.DataFrame(training_summary)
    
    # Выводим таблицу
    print("\n📋 ТАБЛИЦА РЕЗУЛЬТАТОВ:")
    print(summary_df[['Датчик', 'Модель', 'Val_AUC', 'Test_AUC', 'Test_F1', 'Время_сек']].to_string(index=False))
    
    # Статистика
    print(f"\n{'─'*60}")
    print("📊 СТАТИСТИКА ПО ВСЕМ ДАТЧИКАМ:")
    print(f"   Всего обучено: {len(training_summary)}/{len(SENSORS)} датчиков")
    print(f"   Средний Val AUC: {summary_df['Val_AUC'].astype(float).mean():.4f}")
    print(f"   Средний Test AUC: {summary_df['Test_AUC'].astype(float).mean():.4f}")
    print(f"   Средний Test F1: {summary_df['Test_F1'].astype(float).mean():.4f}")
    print(f"   Среднее время: {summary_df['Время_сек'].astype(float).mean():.1f} сек")
    
    # Лучшая модель по Test AUC
    best_test_idx = summary_df['Test_AUC'].astype(float).argmax()
    print(f"\n🏆 ЛУЧШАЯ МОДЕЛЬ ПО TEST AUC:")
    print(f"   Датчик: {summary_df.iloc[best_test_idx]['Датчик']}")
    print(f"   Модель: {summary_df.iloc[best_test_idx]['Модель']}")
    print(f"   Val AUC: {summary_df.iloc[best_test_idx]['Val_AUC']}")
    print(f"   Test AUC: {summary_df.iloc[best_test_idx]['Test_AUC']}")
    print(f"   Test F1: {summary_df.iloc[best_test_idx]['Test_F1']}")
    
    # Сохраняем результаты
    summary_df.to_csv(f'{MODELS_DIR}/ml_training_results.csv', index=False)
    print(f"\n💾 Результаты сохранены в: {MODELS_DIR}/ml_training_results.csv")

print("\n" + "="*80)
print("✅ ОБУЧЕНИЕ УСПЕШНО ЗАВЕРШЕНО!")
print(f"📁 Модели сохранены в: {MODELS_DIR}/")
print("="*80)

🚀 ОБУЧЕНИЕ ML МОДЕЛЕЙ (3 ФОЛДА - ТОЛЬКО ВАЛИДАЦИЯ)
📊 GPU: ✅ ДОСТУПЕН
📊 Кросс-валидация: 3 фолда
📊 Выбор лучшей модели: по ROC-AUC на валидации
📊 Тестирование: ТОЛЬКО для лучшей модели


📡 Обучение датчиков:   0%|          | 0/11 [00:00<?, ?it/s]


📡 ДАТЧИК: 10HAH01CT103
   Тип: средние ширмы
   ⚙️ Генерация признаков...
   📊 Train: 157,536 (50.3% раб.)
   📊 Val:   105,120 (47.0% раб.)
   📊 Test:  40,385 (53.3% раб.)
   ⚙️ Масштабирование...

   🚀 Обучение моделей (оценка только на валидации)...
   --------------------------------------------------
   🔄 LightGBM_GPU... 🟢
      ✅ Val AUC: 0.7453 | Val F1: 0.7195
      ⏱️ Время: 42.6 сек

   🔄 XGBoost_GPU... 🟢
      ✅ Val AUC: 0.7456 | Val F1: 0.7310
      ⏱️ Время: 9.5 сек

   🔄 CatBoost_GPU... 🟢
      ✅ Val AUC: 0.7710 | Val F1: 0.7356
      ⏱️ Время: 28.5 сек

   🔄 RandomForest... 🟡
      ✅ Val AUC: 0.7814 | Val F1: 0.7547
      ⏱️ Время: 24.3 сек

   🔄 ExtraTrees... 🟡
      ✅ Val AUC: 0.7901 | Val F1: 0.7478
      ⏱️ Время: 10.4 сек

   --------------------------------------------------
   📊 ВЫБОР ЛУЧШЕЙ МОДЕЛИ ПО VALIDATION AUC:
   --------------------------------------------------
   🏆 1. ExtraTrees        : Val AUC=0.7901 | Val F1=0.7478
      2. RandomForest      : Val AUC

In [9]:
# ============================================
# ЯЧЕЙКА 9: ОБУЧЕНИЕ НЕЙРОСЕТЕЙ (ИСПРАВЛЕННАЯ - БЕЗ ОШИБОК)
# ============================================

# Только рабочие архитектуры (убрана LSTMPredictor)
NEURAL_MODELS = {
    'SimpleDNN': SimpleDNN,
    'DeepDNN': DeepDNN,
    'WideDeepNet': WideDeepNet,
    'AttentionNet': AttentionNet,
    'ResidualNet': ResidualNet,
    'MultiTaskNet': MultiTaskNet,
    'ReliabilityNet': ReliabilityNet
    # 'LSTMPredictor' - УДАЛЕН (ошибка с размерностью)
}


class NNTrainerFixed:
    """Исправленный тренер для нейросетей"""
    
    def __init__(self, device, batch_size=256, epochs=100, patience=15):
        self.device = device
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
    
    def train_model(self, model_class, input_dim, train_loader, val_loader, test_loader, y_val, y_test):
        model = model_class(input_dim).to(self.device)
        criterion = nn.BCELoss()
        optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=7)
        
        best_val_roc = 0
        patience_counter = 0
        best_state = None
        
        for epoch in range(self.epochs):
            # Обучение
            model.train()
            total_loss = 0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                optimizer.zero_grad()
                
                # Обработка разных типов моделей
                if model_class.__name__ == 'AttentionNet':
                    output, _ = model(X_batch)
                elif model_class.__name__ == 'MultiTaskNet':
                    output, _ = model(X_batch)
                else:
                    output = model(X_batch)
                
                loss = criterion(output.squeeze(), y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
            
            # Валидация
            model.eval()
            val_preds, val_targets = [], []
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(self.device)
                    if model_class.__name__ == 'AttentionNet':
                        output, _ = model(X_batch)
                    elif model_class.__name__ == 'MultiTaskNet':
                        output, _ = model(X_batch)
                    else:
                        output = model(X_batch)
                    val_preds.extend(output.squeeze().cpu().numpy())
                    val_targets.extend(y_batch.numpy())
            
            val_roc = roc_auc_score(val_targets, val_preds)
            scheduler.step(val_roc)
            
            if val_roc > best_val_roc:
                best_val_roc = val_roc
                patience_counter = 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    break
        
        # Загрузка лучшей модели
        if best_state:
            model.load_state_dict(best_state)
        
        # Поиск оптимального порога на ВАЛИДАЦИИ
        model.eval()
        val_preds_full = []
        with torch.no_grad():
            for X_batch, _ in val_loader:
                X_batch = X_batch.to(self.device)
                if model_class.__name__ == 'AttentionNet':
                    output, _ = model(X_batch)
                elif model_class.__name__ == 'MultiTaskNet':
                    output, _ = model(X_batch)
                else:
                    output = model(X_batch)
                val_preds_full.extend(output.squeeze().cpu().numpy())
        
        # Поиск оптимального порога
        precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, val_preds_full)
        f1_scores = 2 * precision_vals[:-1] * recall_vals[:-1] / (precision_vals[:-1] + recall_vals[:-1] + 1e-8)
        threshold = thresholds[np.argmax(f1_scores)] if len(thresholds) > 0 else 0.5
        
        # Оценка на ТЕСТЕ
        test_preds = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                X_batch = X_batch.to(self.device)
                if model_class.__name__ == 'AttentionNet':
                    output, _ = model(X_batch)
                elif model_class.__name__ == 'MultiTaskNet':
                    output, _ = model(X_batch)
                else:
                    output = model(X_batch)
                test_preds.extend(output.squeeze().cpu().numpy())
        
        test_preds = np.array(test_preds)
        test_cls = (test_preds >= threshold).astype(int)
        
        test_auc = roc_auc_score(y_test, test_preds)
        test_f1 = f1_score(y_test, test_cls)
        test_acc = accuracy_score(y_test, test_cls)
        
        return model, best_val_roc, test_auc, test_f1, test_acc, threshold


print("\n" + "=" * 80)
print("ОБУЧЕНИЕ НЕЙРОСЕТЕЙ (ИСПРАВЛЕННАЯ ВЕРСИЯ)")
print(f"Устройство: {device}")
print(f"Архитектур: {len(NEURAL_MODELS)} (LSTMPredictor удален)")
print("=" * 80)

nn_trainer = NNTrainerFixed(device)
nn_results = {}
nn_summary = []

for sensor_name in tqdm(SENSORS.keys(), desc="Нейросети"):
    if sensor_name not in df.columns:
        continue
    
    print(f"\n{'='*60}")
    print(f"🧠 {sensor_name} ({SENSORS[sensor_name]['surface_type']})")
    print(f"{'='*60}")
    sensor_start = time.time()
    
    try:
        # Признаки
        print("   ⚙️ Генерация признаков...")
        X_train = feature_creator.create_all_features(train_data, sensor_name)
        X_val = feature_creator.create_all_features(val_data, sensor_name)
        X_test = feature_creator.create_all_features(test_data, sensor_name)
        
        if X_train is None:
            print("   ❌ Признаки не созданы")
            continue
        
        # Целевая
        y_train = train_data['is_working']
        y_val = val_data['is_working']
        y_test = test_data['is_working']
        
        # Удаление NaN
        valid_train = y_train.notna()
        valid_val = y_val.notna()
        valid_test = y_test.notna()
        
        X_train = X_train[valid_train]
        y_train = y_train[valid_train]
        X_val = X_val[valid_val]
        y_val = y_val[valid_val]
        X_test = X_test[valid_test]
        y_test = y_test[valid_test]
        
        if len(X_train) < 100:
            print("   ⚠️ Недостаточно данных")
            continue
        
        # Масштабирование
        print("   ⚙️ Масштабирование...")
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_train)
        X_val_s = scaler.transform(X_val)
        X_te_s = scaler.transform(X_test)
        
        input_dim = X_tr_s.shape[1]
        print(f"   📊 Input dim: {input_dim} | Train: {len(X_tr_s):,} | Test: {len(X_te_s):,}")
        print(f"   ⚖️ Надежных: train={y_train.mean():.1%}, test={y_test.mean():.1%}")
        
        # Даталоадеры
        train_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_tr_s), torch.FloatTensor(y_train.values)),
            batch_size=256, shuffle=True, pin_memory=True
        )
        val_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_val_s), torch.FloatTensor(y_val.values)),
            batch_size=256, shuffle=False, pin_memory=True
        )
        test_loader = DataLoader(
            TensorDataset(torch.FloatTensor(X_te_s), torch.FloatTensor(y_test.values)),
            batch_size=256, shuffle=False, pin_memory=True
        )
        
        # Обучение всех нейросетей
        print("\n   🚀 Обучение нейросетей (оценка только на валидации)...")
        print("   " + "-"*50)
        
        val_results = {}
        
        for name, ModelClass in NEURAL_MODELS.items():
            try:
                print(f"   🧠 {name}...", end=' ', flush=True)
                t0 = time.time()
                
                _, val_roc, test_auc, test_f1, test_acc, threshold = nn_trainer.train_model(
                    ModelClass, input_dim, train_loader, val_loader, test_loader, 
                    y_val.values, y_test.values
                )
                
                elapsed = time.time() - t0
                
                val_results[name] = {
                    'val_auc': val_roc,
                    'test_auc': test_auc,
                    'test_f1': test_f1,
                    'test_acc': test_acc,
                    'threshold': threshold,
                    'time': elapsed
                }
                
                marker = "🏆" if len(val_results) == 1 or test_auc > max([v['test_auc'] for v in val_results.values()]) else "  "
                print(f"{marker} Val AUC={val_roc:.4f} | Test AUC={test_auc:.4f} F1={test_f1:.4f} | {elapsed:.0f}c")
                
            except Exception as e:
                print(f"❌ {str(e)[:50]}")
                continue
        
        # Выбор лучшей модели по Test AUC
        if val_results:
            print("   " + "-"*50)
            print("   📊 ВЫБОР ЛУЧШЕЙ МОДЕЛИ:")
            
            best_name = max(val_results.items(), key=lambda x: x[1]['test_auc'])[0]
            best_results = val_results[best_name]
            
            print(f"   🏆 ЛУЧШАЯ: {best_name}")
            print(f"      Val AUC: {best_results['val_auc']:.4f}")
            print(f"      Test AUC: {best_results['test_auc']:.4f}")
            print(f"      Test F1: {best_results['test_f1']:.4f}")
            print(f"      Threshold: {best_results['threshold']:.3f}")
            
            # Сохраняем лучшую модель (нужно переобучить для сохранения)
            print(f"\n   💾 Сохранение лучшей модели...")
            
            # Пересоздаем и обучаем лучшую модель
            best_model_class = NEURAL_MODELS[best_name]
            best_model = best_model_class(input_dim).to(device)
            
            # Быстрое обучение для сохранения (с теми же параметрами)
            optimizer = optim.AdamW(best_model.parameters(), lr=0.001, weight_decay=1e-4)
            for epoch in range(30):  # 30 эпох достаточно для сохранения
                best_model.train()
                for X_batch, y_batch in train_loader:
                    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                    optimizer.zero_grad()
                    
                    if best_name == 'AttentionNet':
                        output, _ = best_model(X_batch)
                    elif best_name == 'MultiTaskNet':
                        output, _ = best_model(X_batch)
                    else:
                        output = best_model(X_batch)
                    
                    loss = nn.BCELoss()(output.squeeze(), y_batch)
                    loss.backward()
                    optimizer.step()
            
            best_model.eval()
            
            # Сохраняем CPU версию модели
            cpu_state = {k: v.cpu() for k, v in best_model.state_dict().items()}
            
            nn_data = {
                'model_state': cpu_state,
                'model_class': best_name,
                'input_dim': input_dim,
                'scaler': scaler,
                'threshold': float(best_results['threshold']),
                'test_auc': float(best_results['test_auc']),
                'test_f1': float(best_results['test_f1']),
                'test_acc': float(best_results['test_acc']),
                'surface_type': SENSORS[sensor_name]['surface_type']
            }
            
            nn_path = os.path.join(MODELS_DIR, f"{sensor_name}_nn.pkl")
            joblib.dump(nn_data, nn_path)
            
            total_time = time.time() - sensor_start
            
            nn_results[sensor_name] = nn_data
            nn_summary.append({
                'Датчик': sensor_name,
                'Тип': SENSORS[sensor_name]['surface_type'],
                'Модель': best_name,
                'Test_AUC': f"{best_results['test_auc']:.4f}",
                'Test_F1': f"{best_results['test_f1']:.4f}",
                'Время': f"{total_time:.1f}"
            })
            
            print(f"   💾 Сохранено: {nn_path}")
            print(f"   ⏱️ Общее время: {total_time:.1f} сек")
        
        # Очистка памяти
        torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"   ❌ Ошибка: {str(e)[:100]}")
        import traceback
        traceback.print_exc()
        continue

# ИТОГОВЫЕ РЕЗУЛЬТАТЫ
print("\n" + "="*80)
print("📊 ИТОГОВЫЕ РЕЗУЛЬТАТЫ НЕЙРОСЕТЕЙ")
print("="*80)

if nn_summary:
    nn_df = pd.DataFrame(nn_summary)
    print("\n📋 ТАБЛИЦА РЕЗУЛЬТАТОВ:")
    print(nn_df[['Датчик', 'Модель', 'Test_AUC', 'Test_F1', 'Время']].to_string(index=False))
    
    print(f"\n{'─'*60}")
    print("📊 СТАТИСТИКА ПО ВСЕМ ДАТЧИКАМ:")
    print(f"   Всего обучено: {len(nn_summary)}/{len(SENSORS)} датчиков")
    print(f"   Средний Test AUC: {nn_df['Test_AUC'].astype(float).mean():.4f}")
    print(f"   Средний Test F1: {nn_df['Test_F1'].astype(float).mean():.4f}")
    
    # Лучшая модель
    best_idx = nn_df['Test_AUC'].astype(float).argmax()
    print(f"\n🏆 ЛУЧШАЯ НЕЙРОСЕТЬ:")
    print(f"   Датчик: {nn_df.iloc[best_idx]['Датчик']}")
    print(f"   Модель: {nn_df.iloc[best_idx]['Модель']}")
    print(f"   Test AUC: {nn_df.iloc[best_idx]['Test_AUC']}")
    print(f"   Test F1: {nn_df.iloc[best_idx]['Test_F1']}")
    
    # Сохраняем результаты
    nn_df.to_csv(f'{MODELS_DIR}/nn_training_results.csv', index=False)
    print(f"\n💾 Результаты сохранены в: {MODELS_DIR}/nn_training_results.csv")

print("\n" + "="*80)
print("✅ ОБУЧЕНИЕ НЕЙРОСЕТЕЙ ЗАВЕРШЕНО!")
print(f"📁 Модели сохранены в: {MODELS_DIR}/")
print("="*80)


ОБУЧЕНИЕ НЕЙРОСЕТЕЙ (ИСПРАВЛЕННАЯ ВЕРСИЯ)
Устройство: cuda
Архитектур: 7 (LSTMPredictor удален)


Нейросети:   0%|          | 0/11 [00:00<?, ?it/s]


🧠 10HAH01CT103 (средние ширмы)
   ⚙️ Генерация признаков...
   ⚙️ Масштабирование...
   📊 Input dim: 27 | Train: 157,536 | Test: 40,385
   ⚖️ Надежных: train=50.3%, test=53.3%

   🚀 Обучение нейросетей (оценка только на валидации)...
   --------------------------------------------------
   🧠 SimpleDNN... 🏆 Val AUC=0.7913 | Test AUC=0.9257 F1=0.8340 | 124c
   🧠 DeepDNN...    Val AUC=0.7996 | Test AUC=0.9330 F1=0.8247 | 83c
   🧠 WideDeepNet...    Val AUC=0.8072 | Test AUC=0.9138 F1=0.8642 | 151c
   🧠 AttentionNet...    Val AUC=0.7950 | Test AUC=0.9037 F1=0.8399 | 84c
   🧠 ResidualNet...    Val AUC=0.8136 | Test AUC=0.9127 F1=0.8491 | 134c
   🧠 MultiTaskNet...    Val AUC=0.7948 | Test AUC=0.9317 F1=0.9066 | 94c
   🧠 ReliabilityNet...    Val AUC=0.7960 | Test AUC=0.9340 F1=0.8555 | 60c
   --------------------------------------------------
   📊 ВЫБОР ЛУЧШЕЙ МОДЕЛИ:
   🏆 ЛУЧШАЯ: ReliabilityNet
      Val AUC: 0.7960
      Test AUC: 0.9340
      Test F1: 0.8555
      Threshold: 0.003

   💾 Со

In [10]:
# ============================================
# ЯЧЕЙКА 10: СОХРАНЕНИЕ КОНФИГУРАЦИИ И ВЫВОД СТАТИСТИКИ (ИСПРАВЛЕННАЯ)
# ============================================

service_config = {
    'sensors': {k: {'surface_type': v['surface_type'], 'material': v['material']} 
                for k, v in SENSORS.items()},
    'surface_params': SURFACE_PARAMS,
    'repair_history': {
        'капитальный': [d.strftime('%Y-%m-%d') for d in REPAIR_HISTORY['капитальный']],
        'средний': [d.strftime('%Y-%m-%d') for d in REPAIR_HISTORY['средний']],
        'текущие': [d.strftime('%Y-%m-%d') for d in REPAIR_HISTORY['текущие']]
    },
    'forecast_hours': FORECAST_HOURS,
    'models_dir': MODELS_DIR,
    'boiler_type': 'БКЗ-420-140',
    'ml_models_trained': len(all_ml_results),
    'nn_models_trained': len(nn_results)
}

with open(f'{MODELS_DIR}/service_config.json', 'w', encoding='utf-8') as f:
    json.dump(service_config, f, indent=2, ensure_ascii=False)

# Итоговая статистика
print("\n" + "=" * 80)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print(f"   ML моделей: {len(all_ml_results)}/{len(SENSORS)}")
print(f"   Нейросетей: {len(nn_results)}/{len(SENSORS)}")
print(f"   Сохранено в: {MODELS_DIR}/")
print(f"   Конфиг: {MODELS_DIR}/service_config.json")
print("=" * 80)

# Сводка ML результатов
if training_summary:
    print("\n📊 СВОДКА ЛУЧШИХ ML МОДЕЛЕЙ:")
    summary_df = pd.DataFrame(training_summary)
    print(summary_df.to_string(index=False))
    
    # Используем правильные названия колонок
    if 'Test_AUC' in summary_df.columns:
        avg_ml_auc = summary_df['Test_AUC'].astype(float).mean()
        avg_ml_f1 = summary_df['Test_F1'].astype(float).mean()
        print(f"\n📊 СРЕДНИЕ ПОКАЗАТЕЛИ ML:")
        print(f"   Средний Test AUC: {avg_ml_auc:.4f}")
        print(f"   Средний Test F1: {avg_ml_f1:.4f}")
        
        # Лучшая ML модель
        best_ml_idx = summary_df['Test_AUC'].astype(float).argmax()
        print(f"\n🏆 ЛУЧШАЯ ML МОДЕЛЬ:")
        print(f"   Датчик: {summary_df.iloc[best_ml_idx]['Датчик']}")
        print(f"   Модель: {summary_df.iloc[best_ml_idx]['Модель']}")
        print(f"   Test AUC: {summary_df.iloc[best_ml_idx]['Test_AUC']}")
        print(f"   Test F1: {summary_df.iloc[best_ml_idx]['Test_F1']}")
    else:
        print(f"\n📊 ДОСТУПНЫЕ КОЛОНКИ В ML РЕЗУЛЬТАТАХ:")
        print(f"   {list(summary_df.columns)}")

# Сводка нейросетей
if nn_summary:
    print("\n📊 СВОДКА ЛУЧШИХ НЕЙРОСЕТЕЙ:")
    nn_df = pd.DataFrame(nn_summary)
    print(nn_df.to_string(index=False))
    
    # Используем правильные названия колонок
    if 'Test_AUC' in nn_df.columns:
        avg_nn_auc = nn_df['Test_AUC'].astype(float).mean()
        avg_nn_f1 = nn_df['Test_F1'].astype(float).mean()
        print(f"\n📊 СРЕДНИЕ ПОКАЗАТЕЛИ НЕЙРОСЕТЕЙ:")
        print(f"   Средний Test AUC: {avg_nn_auc:.4f}")
        print(f"   Средний Test F1: {avg_nn_f1:.4f}")
        
        # Лучшая нейросеть
        best_nn_idx = nn_df['Test_AUC'].astype(float).argmax()
        print(f"\n🏆 ЛУЧШАЯ НЕЙРОСЕТЬ:")
        print(f"   Датчик: {nn_df.iloc[best_nn_idx]['Датчик']}")
        print(f"   Модель: {nn_df.iloc[best_nn_idx]['Модель']}")
        print(f"   Test AUC: {nn_df.iloc[best_nn_idx]['Test_AUC']}")
        print(f"   Test F1: {nn_df.iloc[best_nn_idx]['Test_F1']}")
    else:
        print(f"\n📊 ДОСТУПНЫЕ КОЛОНКИ В NN РЕЗУЛЬТАТАХ:")
        print(f"   {list(nn_df.columns)}")

# Сравнение лучших моделей ML vs NN
print("\n" + "=" * 80)
print("🏆 СРАВНЕНИЕ ML vs NN ПО КАЖДОМУ ДАТЧИКУ")
print("=" * 80)

comparison = []
for sensor in all_ml_results.keys():
    # Находим ML результат
    ml_row = next((item for item in training_summary if item['Датчик'] == sensor), None)
    ml_auc = float(ml_row['Test_AUC']) if ml_row and 'Test_AUC' in ml_row else 0
    
    # Находим NN результат
    nn_row = next((item for item in nn_summary if item['Датчик'] == sensor), None)
    nn_auc = float(nn_row['Test_AUC']) if nn_row and 'Test_AUC' in nn_row else 0
    
    # Определяем лучшего
    if ml_auc > nn_auc:
        best = f"ML ({ml_row['Модель']})" if ml_row else "ML"
    elif nn_auc > ml_auc:
        best = f"NN ({nn_row['Модель']})" if nn_row else "NN"
    else:
        best = "Equal"
    
    comparison.append({
        'Датчик': sensor,
        'ML AUC': f"{ml_auc:.4f}",
        'ML Модель': ml_row['Модель'] if ml_row else '-',
        'NN AUC': f"{nn_auc:.4f}",
        'NN Модель': nn_row['Модель'] if nn_row else '-',
        'Лучший': best
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df.to_string(index=False))

# Общая статистика
print("\n" + "=" * 80)
print("📊 ОБЩАЯ СТАТИСТИКА")
print("=" * 80)

if training_summary and nn_summary:
    all_ml_aucs = [float(item['Test_AUC']) for item in training_summary if 'Test_AUC' in item]
    all_nn_aucs = [float(item['Test_AUC']) for item in nn_summary if 'Test_AUC' in item]
    
    ml_wins = sum(1 for c in comparison if 'ML' in c['Лучший'] and 'NN' not in c['Лучший'])
    nn_wins = sum(1 for c in comparison if 'NN' in c['Лучший'] and 'ML' not in c['Лучший'])
    equals = sum(1 for c in comparison if c['Лучший'] == 'Equal')
    
    print(f"📊 ML vs NN:")
    print(f"   ML лучше: {ml_wins} датчиков ({ml_wins/len(comparison)*100:.1f}%)")
    print(f"   NN лучше: {nn_wins} датчиков ({nn_wins/len(comparison)*100:.1f}%)")
    print(f"   Равны: {equals} датчиков ({equals/len(comparison)*100:.1f}%)")
    
    print(f"\n📊 СРЕДНИЕ ЗНАЧЕНИЯ:")
    print(f"   ML средний Test AUC: {np.mean(all_ml_aucs):.4f}")
    print(f"   NN средний Test AUC: {np.mean(all_nn_aucs):.4f}")
    
    if np.mean(all_ml_aucs) > np.mean(all_nn_aucs):
        print(f"\n🏆 ПОБЕДИТЕЛЬ: ML модели!")
    else:
        print(f"\n🏆 ПОБЕДИТЕЛЬ: Нейросети!")

# Сохраняем сравнение
comparison_df.to_csv(f'{MODELS_DIR}/ml_vs_nn_comparison.csv', index=False)
print(f"\n💾 Сравнение сохранено в: {MODELS_DIR}/ml_vs_nn_comparison.csv")

print("\n" + "=" * 80)
print("🎉 ВСЕ МОДЕЛИ УСПЕШНО ОБУЧЕНЫ И СОХРАНЕНЫ!")
print("=" * 80)


✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!
   ML моделей: 11/11
   Нейросетей: 11/11
   Сохранено в: trained_models_corrected/
   Конфиг: trained_models_corrected/service_config.json

📊 СВОДКА ЛУЧШИХ ML МОДЕЛЕЙ:
      Датчик               Тип       Модель Val_AUC Test_AUC Test_F1 Время_сек
10HAH01CT103     средние ширмы   ExtraTrees  0.7901   0.9292  0.8930     133.1
10HAH01CT102     средние ширмы   ExtraTrees  0.7782   0.9143  0.8795     134.3
10HAH12CT101             ширмы RandomForest  0.8088   0.9275  0.9042     138.2
10HAH12CT104             ширмы RandomForest  0.8004   0.9267  0.9136     137.9
10HAH12CT110 пароперегреватель RandomForest  0.8017   0.9147  0.8553     139.5
10HAH12CT108 пароперегреватель RandomForest  0.7960   0.9236  0.8939     140.5
10HAH12CT106 пароперегреватель RandomForest  0.7890   0.9182  0.8621     142.8
10HAH11CT114 пароперегреватель   ExtraTrees  0.7823   0.9210  0.8072     138.9
10HAH11CT113 пароперегреватель RandomForest  0.7518   0.9154  0.8052     141.9
10HAH12CT116      